In [ ]:
# test from (1 ... 18)
n_images = {n_images_value}

# test (True, False)
gwcs_to_fits_sip = {gwcs_to_fits_sip_value}

# server size (small: 2 CPU + 16 GB RAM, medium: 8 CPU + 65 GB RAM)

In [ ]:
from astroquery.mast.missions import MastMissions
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u
from glob import glob

In [ ]:
# Use MAST test to access simulated Roman observations:
def use_mast_test(mast):
    mast._service_api_connection.SERVICE_URL = 'https://masttest.stsci.edu'
    mast._service_api_connection.REQUEST_URL = 'https://masttest.stsci.edu/search/roman/api/v0.1/'
    mast._service_api_connection.MISSIONS_DOWNLOAD_URL = 'https://masttest.stsci.edu/search/'

mm = MastMissions(mission='roman')
use_mast_test(mm)

In [ ]:
# Check for locally cached files for L2s from 18 Roman detectors. If none are found, download them.
existing_downloads = glob(
    'mastDownload/roman/r0000101001001001001_0002*/*'
)

if len(existing_downloads) != 18:
    coord = SkyCoord(ra=270.019512 * u.deg, dec=65.9667761 * u.deg)
    multiple_observations = mm.query_criteria(
        coordinates=coord, 
        radius=30*u.arcmin, 
        optical_element='F158', 
        product_type='l2', 
        exposure_type='WFI_IMAGE'
    )
    mask = ['r0000101001001001001_0002_wfi' in obs['fileSetName'] for obs in multiple_observations]
    one_exposure = multiple_observations[mask]
    assert len(one_exposure) == 18
    product_list = mm.get_product_list(one_exposure)
    cal_files = mm.filter_products(product_list, file_suffix='_cal')
    downloads = mm.download_products(cal_files)
else: 
    downloads = {'Local Path': [path for path in existing_downloads]}

In [ ]:
# Initialize Imviz:
from jdaviz import Imviz

viz = Imviz()
viewer = viz.default_viewer._obj.glue_viewer

viz.show("sidecar:split-right")
# viz.show()

In [ ]:
# Batch load `n_images` into Imviz using `gwcs_to_fits_sip`
sorted(downloads['Local Path'])[:n_images]

In [ ]:
with viz.batch_load():
    for path in sorted(downloads['Local Path'])[:n_images]:
        viz.load(path, format='Image', gwcs_to_fits_sip=gwcs_to_fits_sip)

In [ ]:
# make out-of-bound regions transparent (this needs to be a separate cell for now)
for layer in viewer.state.layers:
    layer.cmap_bad = (0, 0, 0, 0)

In [ ]:
# WCS link:
orientation = viz.plugins['Orientation']
orientation.align_by = 'WCS'

In [ ]:
# Effectively: press the Home button in the viewer toolbar
viewer.reset_limits()